# Proyecto Final Integrador: Riesgo de deserción estudiantil
**Nivel:** trainee · **Metodología:** CRISP-DM · **Fuente:** UCI Machine Learning Repository, *Predict Students' Dropout and Academic Success*.

**Objetivo:** predecir el abandono a partir de datos disponibles al finalizar el primer semestre, para orientar acciones de acompañamiento. Este cuaderno descarga los datos al ejecutarse; **no presenta resultados inventados**.

## 1. Comprensión del negocio
La institución desea identificar estudiantes con riesgo de abandono para ofrecer tutorías y seguimiento. KPI de negocio propuesto: tasa de abandono de estudiantes atendidos. KPI del modelo: **recall de la clase abandono**, acompañado de precisión, F1 y ROC-AUC.

**Alcance:** comparación entre quienes abandonaron (`Dropout`) y quienes se graduaron (`Graduate`). Se excluye `Enrolled` porque su resultado final no está confirmado; esta decisión limita la aplicación directa del modelo a alumnos actualmente cursando.

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, precision_recall_curve

URL = 'https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip'
response = requests.get(URL, timeout=45)
response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    archivos = [x for x in z.namelist() if x.lower().endswith('.csv')]
    print('Archivos CSV:', archivos)
    if not archivos: raise ValueError('No se encontró CSV en el ZIP de UCI')
    with z.open(archivos[0]) as f:
        df = pd.read_csv(f, sep=';')
print('Dimensiones:', df.shape)
display(df.head())

## 2. Comprensión y calidad de los datos
Se revisan filas, columnas, distribución de la variable objetivo, nulos y duplicados. No se modifican valores sin revisar primero su significado.

In [ ]:
print('Columnas:', df.columns.tolist())
print('Duplicados:', df.duplicated().sum())
print('Nulos por columna (mayores primero):')
display(df.isna().sum().sort_values(ascending=False).head(15))
print('Distribución de Target:')
print(df['Target'].value_counts(dropna=False))
df['Target'].value_counts().plot(kind='bar', title='Situación académica')
plt.ylabel('Cantidad de estudiantes'); plt.tight_layout(); plt.show()

In [ ]:
# Revisión adicional de calidad y estadísticas descriptivas
print('Porcentaje de valores faltantes:')
display((df.isna().mean()*100).sort_values(ascending=False).head(10).round(2))
print('Descripción de variables numéricas:')
display(df.describe().T[['mean','std','min','max']].head(15).round(2))
# Esta exploración describe asociaciones, NO relaciones causales.
for variable in ['Age at enrollment','Curricular units 1st sem (approved)']:
    if variable in df.columns:
        df.boxplot(column=variable, by='Target', figsize=(7,4))
        plt.title(f'{variable} según situación académica')
        plt.suptitle(''); plt.ylabel(variable); plt.tight_layout(); plt.show()

### Evidencia adicional del EDA
Esta sección documenta la distribución de `Target`, la calidad de datos y las materias aprobadas en el primer semestre. Las figuras se guardan en `figuras/`. Las cifras son descriptivas y no demuestran causalidad.

In [ ]:
from pathlib import Path
figdir = Path('figuras')
figdir.mkdir(exist_ok=True)
print('Dimensiones:', df.shape)
print('Distribución de Target:')
print(df['Target'].value_counts())
print('Valores faltantes:', int(df.isna().sum().sum()))
print('Duplicados:', int(df.duplicated().sum()))

orden = ['Dropout', 'Enrolled', 'Graduate']
conteos = df['Target'].value_counts().reindex(orden)
plt.figure(figsize=(8, 4.5))
conteos.plot(kind='bar')
plt.title('Distribución de la situación académica')
plt.xlabel('Situación académica')
plt.ylabel('Cantidad de estudiantes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(figdir/'figura_1_distribucion_target.png', dpi=180)
plt.show()

columna = 'Curricular units 1st sem (approved)'
print('Promedio de materias aprobadas por grupo:')
print(df.groupby('Target')[columna].mean().round(2))
datos_cajas = [df.loc[df['Target'] == grupo, columna].dropna() for grupo in orden]
plt.figure(figsize=(9, 5))
plt.boxplot(datos_cajas, labels=orden)
plt.title('Materias aprobadas en el primer semestre')
plt.xlabel('Situación académica')
plt.ylabel('Cantidad de materias aprobadas')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(figdir/'figura_2_materias_aprobadas.png', dpi=180)
plt.show()

## 3. Preparación
Se utilizan variables académicas y de contexto disponibles **hasta el primer semestre**. Se excluyen explícitamente los indicadores del segundo semestre para evitar utilizar información futura. Las variables categóricas del conjunto UCI ya están codificadas numéricamente; para esta comparación inicial se usan tal como están, una simplificación que deberá revisarse en una versión posterior.

In [ ]:
datos = df[df['Target'].isin(['Dropout', 'Graduate'])].copy()
y = (datos['Target'] == 'Dropout').astype(int)
columnas_excluir = ['Target'] + [c for c in datos.columns if '2nd sem' in c.lower()]
X = datos.drop(columns=columnas_excluir).select_dtypes(include='number')
print('Estudiantes con resultado conocido:', len(datos))
print('Variables utilizadas:', X.shape[1])
print('Prevalencia de abandono:', round(y.mean(), 3))
print('Variables:', X.columns.tolist())
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print('Entrenamiento:', len(X_train), '| Prueba:', len(X_test))

## 4. Modelado y validación
Se comparan **Regresión Logística** y **Random Forest**, ambos mediante validación cruzada de cinco particiones solo en el conjunto de entrenamiento. El conjunto de prueba queda reservado para la evaluación final.

In [ ]:
# Baseline sencillo: predecir siempre la clase más frecuente.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train,y_train)
pred_base = baseline.predict(X_test)
print('Baseline (clase mayoritaria):', {'accuracy':round(accuracy_score(y_test,pred_base),3),
  'precision':round(precision_score(y_test,pred_base,zero_division=0),3),
  'recall':round(recall_score(y_test,pred_base,zero_division=0),3),
  'f1':round(f1_score(y_test,pred_base,zero_division=0),3)})

In [ ]:
modelos = {
 'Regresión logística': Pipeline([
     ('imputar', SimpleImputer(strategy='median')),
     ('escalar', StandardScaler()),
     ('modelo', LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42))]),
 'Random Forest': Pipeline([
     ('imputar', SimpleImputer(strategy='median')),
     ('modelo', RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                     class_weight='balanced', random_state=42))])
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
filas=[]
for nombre, modelo in modelos.items():
    puntajes = cross_validate(modelo, X_train, y_train, cv=cv,
      scoring={'recall':'recall', 'precision':'precision', 'f1':'f1', 'auc':'roc_auc'}, n_jobs=-1)
    filas.append({'Modelo':nombre, **{m:puntajes['test_'+m].mean() for m in ['recall','precision','f1','auc']}})
resultados = pd.DataFrame(filas).sort_values(['recall','f1'], ascending=False)
display(resultados.round(3))

## 5. Evaluación final
**Regla de elección propuesta:** elegir mayor recall en validación cruzada, usando F1 para desempatar. El umbral inicial es 0,50; debe ajustarse según la capacidad real de tutorías y el costo de falsas alarmas.

In [ ]:
nombre_ganador = resultados.iloc[0]['Modelo']
mejor = modelos[nombre_ganador]
mejor.fit(X_train, y_train)
pred = mejor.predict(X_test)
prob = mejor.predict_proba(X_test)[:,1]
print('Modelo elegido:', nombre_ganador)
print(classification_report(y_test, pred, target_names=['Graduate', 'Dropout'], zero_division=0))
print('ROC-AUC:', round(roc_auc_score(y_test, prob), 3))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['Graduate','Dropout'])
plt.title('Matriz de confusión - conjunto de prueba');plt.tight_layout();plt.show()

## 6. Impacto de negocio y conclusiones
Una institución puede revisar los casos señalados y ofrecer orientación académica, pero **no debe tomar decisiones automáticas sobre estudiantes**. Antes de implementar, validar con datos de la institución, medir resultados de tutorías y revisar posibles sesgos por características socioeconómicas.

**Limitaciones:** datos de otra institución/país; solo se modelan resultados conocidos; codificaciones categóricas tratadas como numéricas; ausencia de validación temporal y de evaluación de equidad. Las métricas que se obtengan al ejecutar este notebook son experimentales, no una garantía de desempeño futuro.

## 7. Exportación de resultados y límites
Se guardan las métricas reales de la ejecución. **No incluir resultados en el informe antes de correr todas las celdas.** La muestra excluye a estudiantes con estado *Enrolled* y los resultados no equivalen a una validación prospectiva. Se usan datos del primer semestre; por tanto, el sistema no sirve para detectar riesgo antes de comenzar el curso. Las variables personales requieren revisión ética antes de cualquier uso institucional.

In [ ]:
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
salida=Path('resultados');salida.mkdir(exist_ok=True)
resultados.to_csv(salida/'comparacion_validacion_cruzada.csv',index=False)
resumen=pd.DataFrame([{'modelo':nombre_ganador,
 'accuracy':accuracy_score(y_test,pred), 'precision':precision_score(y_test,pred,zero_division=0),
 'recall':recall_score(y_test,pred,zero_division=0), 'f1':f1_score(y_test,pred,zero_division=0),
 'roc_auc':roc_auc_score(y_test,prob), 'n_test':len(y_test)}])
resumen.to_csv(salida/'metricas_test.csv',index=False)
pd.DataFrame(confusion_matrix(y_test,pred),index=['Real graduado','Real abandono'],
 columns=['Predicho graduado','Predicho abandono']).to_csv(salida/'matriz_confusion.csv')
print('Archivos generados en:',salida.resolve())
display(resumen.round(3))